#### Visão Geral
##### Schema : bronze
##### Table : logistica_entregas

| Detalhe | Informação |
|---------|------------|
| Criado Originalmente Por | Wellikiandre Bosich |
| Tabela de Dados de Saída | `{environment}.bronze.nomearquivo` |
| Origem Fonte de Dados de Entrada | Camada Landing |
| Destino Fonte de Dados de Saída | Camada bronze |

#### Histórico

| Data       | Desenvolvido Por         | Motivo                                         |
|:----------:|--------------------------|-----------------------------------------------|
| 04/06/2026 | Wellikiandre Bosich    | Criação do notebook e ajuste de códigos para padronização/replicação da camada Bronze. |

In [0]:
%run ../../0_Config/0-Init

In [0]:
# Parâmetros de Inicialização
sistema = 'case'
table_name = 'logistica_entregas'
input_path = f"{var_landing}/{sistema}/{table_name}/{table_name}*"
output_path_data = f"{var_bronze}/{sistema}/{table_name}/data"
output_path_schema = f"{var_bronze}/{sistema}/{table_name}/_schemalocal"
output_path_checkpoint = f"{var_bronze}/{sistema}/{table_name}/_checkpoint"
table_name_schema = f'{var_environment}.{var_bronze_schema}.{sistema}_{table_name}'

print(f"Parâmetros de Inicialização:")
print(f"table_name: {table_name}")
print(f"input_path: {input_path}")
print(f"output_path_data: {output_path_data}")
print(f"output_path_schema: {output_path_schema}")
print(f"output_path_checkpoint: {output_path_checkpoint}")
print(f"table_name_schema: {table_name_schema}")

In [0]:
from pyspark.sql.functions import col, current_timestamp, to_utc_timestamp, from_utc_timestamp

dfRead = (
    spark.read.format('json')
    .option('multiline', 'true')
    .load(input_path)
    .withColumn('rastreamento_source', col('_metadata.file_path'))
    .withColumn('ingestion_date_brasilia', from_utc_timestamp(to_utc_timestamp(current_timestamp(), 'UTC'), 'America/Sao_Paulo'))
    .select(
        col("carrier.mode").alias("carrier_mode"),
        col("carrier.name").alias("carrier_name"),
        col("cost").alias("cost"),
        col("delivery_id"),
        col("delivery_status"),
        col("destination.city").alias("destination_city"),
        col("destination.state").alias("destination_state"),
        col("order_ref"),
        col("timestamps.delivered_at").alias("timestamps_delivered_at"),
        col("timestamps.shipped_at").alias("timestamps_shipped_at"),
        col("rastreamento_source"),
        col("ingestion_date_brasilia")
    )
)

dfRead.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(output_path_data)